# **Eksplorasi dan Persiapan Data**

## 0. Import Library

In [1]:
import pandas as pd
import geopandas as gpd

## 1. Data Acquisition
### a. Sumber dataset
Dataset yang digunakan adalah **Groundsource: A Dataset of Flood Events from News (v1)**, dipublikasikan di Zenodo (15 Februari 2026) dengan DOI **10.5281/zenodo.18647054**. Dataset ini berisi sekitar **2,6 juta** catatan kejadian banjir global dari >150 negara.

### b. Cara memperoleh data
Data diunduh langsung dari portal open-source Zenodo: `https://zenodo.org/records/18647054` dengan berkas mentah `groundsource_2026.parquet` (sekitar 667,1 MB). Fokus editorialnya adalah Indonesia, jadi data global kemudian difilter secara spasial agar hanya menyisakan kejadian pada wilayah Indonesia.

### c. Lisensi dan etika penggunaan
Dataset bersifat terbuka dengan lisensi **CC BY 4.0**. Ini memperbolehkan penggunaan, distribusi, dan modifikasi data untuk kebutuhan akademik dengan syarat atribusi. Sitasi kreator asli (*Mayo, R., Zlydenko, O., dkk., 2026*) wajib dicantumkan pada laporan dan visualisasi final.

### d. Data preprocessing dari file hasil unduhan
Setelah file mentah diunduh, dilakukan preprocessing bertahap:
1. Membaca berkas mentah parquet global.
2. Mengonversi geometri WKB menjadi geometri spasial yang dapat diproses GeoPandas.
3. Melakukan *spatial filtering* untuk menyisakan record yang berada di Indonesia.
4. Menyimpan versi hasil preprocessing (`groundsource_indonesia.parquet`) agar analisis berikutnya lebih ringan dan reproducible.

In [2]:
from pathlib import Path

# Data Acquisition + preprocessing dari file unduhan
raw_path = Path('../data/raw/groundsource_2026.parquet')
print("1) Membaca file parquet mentah (global)...")
df_raw = pd.read_parquet(raw_path)
raw_size_mb = raw_path.stat().st_size / (1024 * 1024)
print(f"   Ukuran file mentah: {raw_size_mb:.1f} MB")
print(f"   Total record global: {len(df_raw):,}")

print("2) Mengonversi kolom geometri ke format spasial...")
df_raw['geometry'] = gpd.GeoSeries.from_wkb(df_raw['geometry'])
gdf_global = gpd.GeoDataFrame(df_raw, geometry='geometry', crs='EPSG:4326')

print("3) Memuat batas negara dan memilih poligon Indonesia...")
url_peta_dunia = 'https://raw.githubusercontent.com/johan/world.geo.json/master/countries.geo.json'
world = gpd.read_file(url_peta_dunia)
indonesia_map = world[world['name'] == 'Indonesia']

print("4) Spatial join: memfilter kejadian banjir di wilayah Indonesia...")
gdf_indo = gpd.sjoin(gdf_global, indonesia_map, predicate='intersects')

kolom_penting = ['uuid', 'area_km2', 'geometry', 'start_date', 'end_date']
gdf_indo_raw = gdf_indo[kolom_penting].copy()
print(f"   Total record Indonesia (hasil filter): {len(gdf_indo_raw):,}")

out_path = Path('../data/processed/groundsource_indonesia.parquet')
print("5) Menyimpan hasil preprocessing Indonesia...")
gdf_indo_raw.to_parquet(out_path, index=False)
out_size_mb = out_path.stat().st_size / (1024 * 1024)
rasio = (out_size_mb / raw_size_mb) if raw_size_mb > 0 else 0.0
print(f"   Ukuran file hasil preprocessing: {out_size_mb:.1f} MB")
print(f"   Rasio ukuran hasil/raw: {rasio:.3f}")
print("   Selesai: data akuisisi + preprocessing tersimpan.")

1) Membaca file parquet mentah (global)...
   Ukuran file mentah: 636.2 MB
   Total record global: 2,646,302
2) Mengonversi kolom geometri ke format spasial...
3) Memuat batas negara dan memilih poligon Indonesia...
4) Spatial join: memfilter kejadian banjir di wilayah Indonesia...
   Total record Indonesia (hasil filter): 310,731
5) Menyimpan hasil preprocessing Indonesia...
   Ukuran file hasil preprocessing: 51.8 MB
   Rasio ukuran hasil/raw: 0.081
   Selesai: data akuisisi + preprocessing tersimpan.


## 2. Data Examination
Pemeriksaan dilakukan pada dua aspek utama, yaitu **completeness** (kelengkapan) dan **quality** (kualitas). Fokus pemeriksaannya berupa dimensi data, missing values, duplikasi, validitas tanggal, validitas nilai area, dan geometri yang tidak valid.

In [3]:
# Data Examination
print("=" * 60)
print("DATA EXAMINATION")
print("=" * 60)

if 'gdf_indo_raw' not in globals():
    gdf_indo_raw = gpd.read_parquet('../data/processed/groundsource_indonesia.parquet')

print("\n[A] Struktur dan Tipe Data")
print(f"Dimensi (baris, kolom): {gdf_indo_raw.shape}")
display(gdf_indo_raw.head(3))
gdf_indo_raw.info()

print("\n[B] Completeness Check")
missing_counts = gdf_indo_raw.isnull().sum()
print("Missing values per kolom:")
print(missing_counts)

periode_start = pd.to_datetime(gdf_indo_raw['start_date'], errors='coerce')
periode_end = pd.to_datetime(gdf_indo_raw['end_date'], errors='coerce')
print(f"Rentang start_date: {periode_start.min()} s.d. {periode_start.max()}")
print(f"Rentang end_date  : {periode_end.min()} s.d. {periode_end.max()}")

print("\n[C] Quality Check")
duplikat = gdf_indo_raw.duplicated(subset=['start_date', 'end_date', 'area_km2', 'geometry']).sum()
area_negatif = (gdf_indo_raw['area_km2'] < 0).sum()
tanggal_terbalik = (periode_end < periode_start).sum()
geom_invalid = (~gdf_indo_raw.geometry.is_valid).sum()
geom_kosong = gdf_indo_raw.geometry.is_empty.sum()

print(f"Potensi duplikasi: {duplikat:,}")
print(f"Nilai area negatif: {area_negatif:,}")
print(f"Tanggal akhir < tanggal mulai: {tanggal_terbalik:,}")
print(f"Geometri tidak valid: {geom_invalid:,}")
print(f"Geometri kosong: {geom_kosong:,}")

DATA EXAMINATION

[A] Struktur dan Tipe Data
Dimensi (baris, kolom): (310731, 5)


,uuid,area_km2,geometry,start_date,end_date
36,6df5fa6994464a38b9d4ecfa707ccedf,2734.230080,"POLYGON ((96.78598 4.94206, 96.81658 4.98954, ...",2000-04-17,2000-04-17
48,1c561141acfb4a5a9f1ef6cc5408160c,76.443643,"POLYGON ((124.86905 -9.66683, 124.90642 -9.652...",2000-05-16,2000-05-16
548,5e6f06b9a6e94e79bb3a9b20dd469206,50.068620,"POLYGON ((104.16267 -4.92204, 104.19404 -4.862...",2001-01-23,2001-01-23


<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 310731 entries, 36 to 2752389
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype   
---  ------      --------------   -----   
 0   uuid        310731 non-null  object  
 1   area_km2    310731 non-null  float64 
 2   geometry    310731 non-null  geometry
 3   start_date  310731 non-null  object  
 4   end_date    310731 non-null  object  
dtypes: float64(1), geometry(1), object(3)
memory usage: 14.2+ MB

[B] Completeness Check
Missing values per kolom:
uuid          0
area_km2      0
geometry      0
start_date    0
end_date      0
dtype: int64
Rentang start_date: 2000-04-17 00:00:00 s.d. 2026-02-03 00:00:00
Rentang end_date  : 2000-04-17 00:00:00 s.d. 2026-02-03 00:00:00

[C] Quality Check
Potensi duplikasi: 34
Nilai area negatif: 0
Tanggal akhir < tanggal mulai: 0
Geometri tidak valid: 0
Geometri kosong: 0


## 3. Data Type Identification
Klasifikasi tipe data diperlukan agar operasi analisis tepat. Pada konteks ini, variabel waktu diperlakukan sebagai **interval**, metrik luas terdampak sebagai **ratio**, dan pengenal kejadian sebagai **nominal**.

In [ ]:
# Data Type Identification
if 'gdf_indo_raw' not in globals():
    gdf_indo_raw = gpd.read_parquet('../data/processed/groundsource_indonesia.parquet')

data_type_map = pd.DataFrame(
    [
        ('uuid', 'Nominal', 'ID unik kejadian banjir; tidak punya urutan matematis'),
        ('start_date', 'Interval', 'Data waktu; bisa dibandingkan/selisih waktu, bukan rasio absolut'),
        ('end_date', 'Interval', 'Data waktu; dipakai untuk durasi/periode kejadian'),
        ('area_km2', 'Ratio', 'Punya nol absolut; sah untuk +, -, x, /'),
        ('geometry', 'Nominal-Spatial', 'Representasi lokasi spasial untuk analisis geografi')
    ],
    columns=['variabel', 'tipe_data', 'alasan']
 )

print("Klasifikasi tipe data:")
display(data_type_map)

Klasifikasi tipe data:


,variabel,tipe_data,alasan
0,uuid,Nominal,"ID unik kejadian banjir, tidak punya urutan ma..."
1,start_date,Interval,"Data waktu; bisa dibandingkan/selisih waktu, b..."
2,end_date,Interval,Data waktu; dipakai untuk durasi/periode kejadian
3,area_km2,Ratio,"Punya nol absolut; sah untuk +, -, x, /"
4,geometry,Nominal-Spatial,Representasi lokasi spasial untuk analisis geo...


## 4. Data Cleaning & Quality Improvement
Tahap ini menangani isu kualitas yang ditemukan saat examination, seperti duplikasi, missing value, anomali, dan format data waktu. Hasilnya disimpan sebagai dataset bersih.

In [5]:
# Data Cleaning & Quality Improvement
if 'gdf_indo_raw' not in globals():
    gdf_indo_raw = gpd.read_parquet('../data/processed/groundsource_indonesia.parquet')

gdf_clean = gdf_indo_raw.copy()
n_awal = len(gdf_clean)

# Konversi tanggal dengan coercion untuk mengatasi format tidak valid
gdf_clean['start_date'] = pd.to_datetime(gdf_clean['start_date'], errors='coerce')
gdf_clean['end_date'] = pd.to_datetime(gdf_clean['end_date'], errors='coerce')

# Hapus duplikasi
gdf_clean = gdf_clean.drop_duplicates(subset=['start_date', 'end_date', 'area_km2', 'geometry'])

# Vlidaasi baris
gdf_clean = gdf_clean.dropna(subset=['start_date', 'end_date', 'area_km2', 'geometry'])
gdf_clean = gdf_clean[gdf_clean['area_km2'] >= 0]
gdf_clean = gdf_clean[gdf_clean['end_date'] >= gdf_clean['start_date']]
gdf_clean = gdf_clean[gdf_clean.geometry.is_valid]
gdf_clean = gdf_clean[~gdf_clean.geometry.is_empty]

n_akhir = len(gdf_clean)
print(f"Jumlah record sebelum cleaning: {n_awal:,}")
print(f"Jumlah record sesudah cleaning: {n_akhir:,}")
print(f"Record tereliminasi: {n_awal - n_akhir:,}")

gdf_clean.to_parquet('../data/processed/groundsource_indonesia_clean.parquet', index=False)
print("Dataset bersih tersimpan: ../data/processed/groundsource_indonesia_clean.parquet")

Jumlah record sebelum cleaning: 310,731
Jumlah record sesudah cleaning: 310,697
Record tereliminasi: 34
Dataset bersih tersimpan: ../data/processed/groundsource_indonesia_clean.parquet


## 5. Data Transformation for Analysis
Data bersih ditransformasikan agar siap untuk analisis tren: ekstraksi tahun, kalkulasi frekuensi kejadian, total area terdampak, dan durasi median kejadian per tahun. Resolusi yang dipakai adalah **aggregate** per tahun.

In [6]:
# Data Transformation for Analysis
if 'gdf_clean' not in globals():
    gdf_clean = gpd.read_parquet('../data/processed/groundsource_indonesia_clean.parquet')

df_analisis = gdf_clean.copy()
df_analisis['year'] = df_analisis['start_date'].dt.year
df_analisis['durasi_hari'] = (df_analisis['end_date'] - df_analisis['start_date']).dt.days

df_trend = (
    df_analisis.groupby('year').agg(
        frekuensi_banjir=('uuid', 'count'),
        total_area_km2=('area_km2', 'sum'),
        median_durasi_hari=('durasi_hari', 'median')
    ).reset_index().sort_values('year')
 )

display(df_trend.head())
display(df_trend.tail())

df_trend.to_csv('../data/processed/trend_banjir_indonesia_2000_2026.csv', index=False)
print("Dataset tren tahunan tersimpan: ../data/processed/trend_banjir_indonesia_2000_2026.csv")

,year,frekuensi_banjir,total_area_km2,median_durasi_hari
0,2000,2,2810.673723,0.0
1,2001,2,422.438832,0.0
2,2002,26,8314.873103,0.0
3,2003,21,2020.188209,2.0
4,2004,8,1549.349115,0.0


,year,frekuensi_banjir,total_area_km2,median_durasi_hari
22,2022,36675,4.259661e+06,0.0
23,2023,25850,3.184410e+06,0.0
24,2024,43075,5.663502e+06,0.0
25,2025,59246,6.494240e+06,0.0
26,2026,10934,9.272079e+05,0.0


Dataset tren tahunan tersimpan: ../data/processed/trend_banjir_indonesia_2000_2026.csv


## 6. Data Consolidation
Tahap konsolidasi menambahkan data pendukung untuk memperkaya konteks analisis. Pada notebook ini, data kejadian banjir yang sudah bersih dikonsolidasikan dengan batas administrasi provinsi Indonesia melalui *spatial join*, sehingga analisis dapat dilanjutkan ke level wilayah/provinsi.

In [9]:
# Data Consolidation
if 'gdf_clean' not in globals():
    gdf_clean = gpd.read_parquet('../data/processed/groundsource_indonesia_clean.parquet')

url_prov = 'https://raw.githubusercontent.com/superpikar/indonesia-geojson/master/indonesia-province-simple.json'
prov_map = gpd.read_file(url_prov)
if gdf_clean.crs != prov_map.crs:
    prov_map = prov_map.to_crs(gdf_clean.crs)

gdf_prov = gpd.sjoin(gdf_clean, prov_map, predicate='intersects')

# Agregasi tambahan: frekuensi kejadian per provinsi
df_konsolidasi_prov = (
    gdf_prov.groupby('Propinsi')
    .size()
    .reset_index(name='total_kejadian')
    .sort_values('total_kejadian', ascending=False)
 )

display(df_konsolidasi_prov.head(10))
df_konsolidasi_prov.to_csv('../data/processed/banjir_konsolidasi_provinsi.csv', index=False)
print("Dataset konsolidasi tersimpan: ../data/processed/banjir_konsolidasi_provinsi.csv")

,Propinsi,total_kejadian
10,JAWA BARAT,41926
12,JAWA TIMUR,36856
11,JAWA TENGAH,32761
3,DI. ACEH,29247
4,DKI JAKARTA,27312
30,SUMATERA UTARA,18565
22,PROBANTEN,14811
14,KALIMANTAN SELATAN,11662
24,SULAWESI SELATAN,11621
23,RIAU,11373


Dataset konsolidasi tersimpan: ../data/processed/banjir_konsolidasi_provinsi.csv
